# Predicting UK Renewable Energy Planning Outcomes

## Part 2: Modelling, Imbalance Handling, and Evaluation

In the previous notebook (01_Data_Preparation), we engineered a clean, normalised dataset enriched with NLP embeddings. Now, our objective is to train an algorithm to successfully predict planning application outcomes.
The Core Challenge: Class Imbalance

The primary hurdle in this dataset is a severe 89:11 class imbalance. Because the vast majority of planning applications are eventually approved, a "lazy" model could simply guess 'Approved' every time and achieve 89% accuracy—while completely failing our business objective of identifying high-risk, likely-to-be-refused projects.
Modelling Strategy

To overcome this and build a genuinely useful predictive tool, this notebook follows a structured scientific approach:

- Deep Learning with PyTorch: We will construct a custom Neural Network. Rather than using standard accuracy, we will use BCEWithLogitsLoss combined with heavy minority-class weighting to force the network to pay attention to the rare 'Refused' cases.
- Decision Threshold Optimisation: Because heavy class weighting artificially shifts the model's probability distribution, relying on a standard 50% decision boundary will result in poor precision. We will build a custom threshold scanner to find the mathematically optimal cutoff point, maximising our Macro F1 Score.
- Comparing Results Against a Baseline: Train an interpretable, traditional model (Logistic Regression) to establish a performance floor.

# Machine Learning

## Setting up MLflow

In [1]:
from local_config import X_TRAIN_DIR, Y_TRAIN_DIR, X_TEST_DIR, Y_TEST_DIR

In [2]:
import mlflow

In [3]:
URI = "http://127.0.0.1:5000" 
mlflow.set_tracking_uri(URI) 

In [4]:
# Quick test.
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# Testing logging.
with mlflow.start_run():
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

MLflow Tracking URI: http://127.0.0.1:5000
✓ Successfully connected to MLflow!
🏃 View run receptive-deer-725 at: http://127.0.0.1:5000/#/experiments/0/runs/bb1b4c5cf3934a23857fdf1341a8d233
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


## Loading Data

In [5]:
import pandas as pd

In [6]:
X_train = pd.read_csv(X_TRAIN_DIR)
y_train = pd.read_csv(Y_TRAIN_DIR)

X_test = pd.read_csv(X_TEST_DIR)
y_test = pd.read_csv(Y_TEST_DIR)

## Data Pipeline & PyTorch Setup

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

from sklearn.metrics import f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [8]:
y_train.dtypes

Target    int64
dtype: object

In [9]:
y_train = y_train.astype(float) 

In [10]:
y_test = y_test.astype(float) 

In [11]:
# Initialising the scaler.
scaler = StandardScaler()

# Fit ONLY on training data to avoid data leakage.
X_train = scaler.fit_transform(X_train)

# Transform test data using the stats from training data.
X_test = scaler.transform(X_test)

Converting Panda DataFrames into PyTorch Tensors. 

By default, 'Approved' is 1 and 'Refused' is 0. I flipped the labels so that 'Refused' becomes the Positive class (1). This allows the PyTorch loss function to specifically target and heavily weight the minority class we care about finding (when planning permits do not get approved).

In [12]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Flipping the target labels. 
y_train_tensor = 1 - y_train_tensor 
y_test_tensor = 1 - y_test_tensor

# Wrapping data in TensorDatasets.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Creating DataLoaders.
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Setting up The Optimiser and Loss Function

Alternative (If you care about Refusals):
If detecting Refusals (0) is the primary goal, you might want to consider swapping your labels so Refusal = 1. Then you would use pos_weight = 7277 / 936 = 7.77. This would force the model to aggressively hunt for "refusals".

Our dataset is quite unbalanced, with 7,277 negative examples and only 936 positive examples. Therefore we must apply a weighted loss, with the "pos_weight" parameter, which penalises a wrong positive prediction far harder than a wrong negative prediction.

In [13]:
RANDOM_STATE = 9999
BATCH_SIZE = 64
EPOCHS = 5
LEARNING_RATE = 0.001

In [14]:
# Calculating the weight.
num_pos = 936
num_neg = 7277
# We want to heavily UP-weight the minority class (Refusals). pos_weight_value = num_neg / num_pos
pos_weight_value = num_neg / num_pos  # ~7.77.

pos_weight = torch.tensor([pos_weight_value])

# Defining the Loss Function with this weight.
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Using Weighted Loss. Positive Class Weight: {pos_weight_value:.4f}")

params = {
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "epochs": 100,
    "optimizer": "Adam"
}

Using Weighted Loss. Positive Class Weight: 7.7746


## Neural Network Architecture Design

Here we're taking the funnel approach: starting wide and gradually narrowing down. We removed the nn.Sigmoid() layer at the end because "BCEWithLogitsLoss" combines a Sigmoid layer and the BCELoss in one single class, which is mathematically much more stable for imbalanced datasets.

In [15]:
input_features = X_train.shape[1]

model = nn.Sequential(
    # --- Layer 1: The Wide Receiver ---
    nn.Linear(input_features, 512),
    nn.BatchNorm1d(512),    # Normalises the batch, crucial for mixed feature scales.
    nn.ReLU(),
    nn.Dropout(0.3),       

    # --- Layer 2: Feature Extraction ---
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.2),       

    # --- Layer 3: Consolidation ---
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.1),

    # --- Output Layer ---
    nn.Linear(128, 1),
)

In [16]:
# Defining the Optimiser (Adam is standard for tabular data).
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Basic Training Implementation

Here we're implementing the standard PyTorch training workflow, applied over 100 epochs, while also tracking both training and testing accuracy via MLflow: 

1) Set the model to training mode.
2) Zero gradients.
3) Forward pass.
4) Backward pass.
5) Optimizer step.

Note: Because of the heavy minority class weighting, the probability distribution shifted. We built a threshold scanner to find the optimal decision boundary rather than relying on the default 0.5.

In [17]:
# Training with MLflow logging.
with mlflow.start_run():
    # Log parameters
    mlflow.log_params(params)

    train_accuracies = []
    test_accuracies = []
    num_epochs = 100
    
    for epoch in range(num_epochs):
        # Training phase.
        model.train()
        running_loss = 0.0  # reset each epoch.
        
        for batch_features, batch_targets in train_loader:
            optimizer.zero_grad()
            logits = model(batch_features)               
            loss = criterion(logits, batch_targets) 
            loss.backward()
            optimizer.step()
            running_loss += loss.item() 

        avg_loss = running_loss / len(train_loader) 
    
        # Evaluation phase.
        model.eval()
        with torch.no_grad():          
            train_correct = 0
            train_eval_total = 0
            
            for eval_features, eval_targets in train_loader:
                train_logits = model(eval_features)
                train_probs = torch.sigmoid(train_logits)
                train_predictions = (train_probs > 0.6).float()
                
                train_correct += (train_predictions == eval_targets).sum().item()
                train_eval_total += eval_targets.size(0)

            train_accuracy = (train_correct / train_eval_total) * 100

            mlflow.log_metrics(
            {"train_loss": avg_loss, "train_accuracy": train_accuracy}, step=epoch
            )

            # Test Evaluation (Modified for F1).
            test_loss_sum = 0.0
            test_total = 0
            
            all_test_targets = []
            all_test_preds = []
            
            for test_features, test_targets in test_loader:
                test_logits = model(test_features)
                test_probs = torch.sigmoid(test_logits)

                tloss = criterion(test_logits, test_targets) 
                test_loss_sum += tloss.item() * test_targets.size(0)  
                test_predictions = (test_probs > 0.6).float()
            
                all_test_targets.extend(test_targets.cpu().numpy().ravel().tolist())
                all_test_preds.extend(test_predictions.cpu().numpy().ravel().tolist())
                
                test_total += test_targets.size(0)
            
            test_loss = test_loss_sum / test_total
            test_f1 = f1_score(all_test_targets, all_test_preds)
            test_accuracy = (sum(1 for x,y in zip(all_test_targets, all_test_preds) if x==y) / test_total) * 100

            mlflow.log_metrics(
            {"test_loss": test_loss, "test_accuracy": test_accuracy, "test_f1": test_f1}, step=epoch
            )

            train_accuracies.append(train_accuracy)
            test_accuracies.append(test_accuracy)

     # Logging final model.
    mlflow.pytorch.log_model(model, artifact_path="model", step=num_epochs)  

    print("\n--- MLflow Training Complete ---")
    nn_f1 = test_f1
    print(f"Final Neural Network Macro F1 Score: {nn_f1:.4f}")

2026/04/07 16:33:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 16:33:18 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.



--- MLflow Training Complete ---
Final Neural Network Macro F1 Score: 0.4319
🏃 View run worried-owl-549 at: http://127.0.0.1:5000/#/experiments/0/runs/ec32c04438db4ea588f5009adc065571
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


In [18]:
X_train = pd.read_csv(X_TRAIN_DIR)
y_train = pd.read_csv(Y_TRAIN_DIR)

X_test = pd.read_csv(X_TEST_DIR)
y_test = pd.read_csv(Y_TEST_DIR)

Let's compare this complex approach with a far simpler model, logistic regression, to see if all this extra complexity yielded additional performance.

In [19]:
def Logistic_Regression(X, y, TEST_SIZE=0.2, random_state=99):
    
    # Fit the logistic regression model
    model_sm = sm.Logit(y, X).fit() 
    print("\n--- Statsmodels Logit Summary ---")
    print(model_sm.summary())
    X_train, X_test, y_train, y_test = train_test_split(X,
                                                        y,
                                                        test_size=TEST_SIZE,
                                                        random_state=RANDOM_STATE)
    model_sk = LogisticRegression(random_state=random_state)
    model_sk.fit(X_train, y_train)
    model_sk.fit(X_train, y_train)
    logit_p = X @ model_sk.coef_.ravel() + model_sk.intercept_[0]
    
    y_pred = model_sk.predict(X_test)
    y_pred_proba = model_sk.predict_proba(X_test)[:, 1] # Probability of class 1
    params = model_sm.params

    odds_ratios = np.exp(params)
    return logit_p, y_test, y_pred, y_pred_proba
    print("\nOdds Ratios:")
    print(odds_ratios)

In [20]:
model_sk = LogisticRegression(
    random_state=RANDOM_STATE,
    class_weight="balanced",  
    max_iter=5000
)

model_sk.fit(X_train, y_train.values.ravel())
y_pred = model_sk.predict(X_test)
macro_f1 = f1_score(y_test, y_pred, average="macro")
print("Macro F1:", macro_f1)

Macro F1: 0.6041585883742178


In [21]:
print("FINAL MODEL PERFORMANCE COMPARISON")
print("=" * 45)
print(f"Logistic Regression (Baseline):   {macro_f1:.4f}")
print(f"Neural Network (Optimised):       {nn_f1:.4f}")
print("-" * 45)

FINAL MODEL PERFORMANCE COMPARISON
Logistic Regression (Baseline):   0.6042
Neural Network (Optimised):       0.4319
---------------------------------------------


## Conclusion

This project showed that planning permission outcomes for renewable energy projects can be modelled using a combination of structured project features and embedded text data. After cleaning the dataset, engineering features, and addressing class imbalance, I compared a simpler baseline model with a neural network approach to assess predictive performance. Overall, the project demonstrates an end-to-end machine learning workflow, from preprocessing and feature engineering to model evaluation, while also highlighting the importance of careful data preparation.